# Installing required Python modules

In [ ]:
# After executing the following commands you will need to restart the Python kernal (from the Kernel menu).
%pip install ipyleaflet
%pip install gtfs-realtime-bindings

In [72]:
import math

# https://en.wikipedia.org/wiki/Haversine_formula
def haversine_distance(lon1, lat1, lon2, lat2):
      # convert decimal degrees to radians 
      lon1 = math.radians(lon1)
      lat1 = math.radians(lat1)
      lon2 = math.radians(lon2)
      lat2 = math.radians(lat2)
        
      # haversine formula 
      dlon = lon2 - lon1 
      dlat = lat2 - lat1 
      a =  math.sin(dlat/2)**2 +  math.cos(lat1) * math.cos(lat2) *  math.sin(dlon/2)**2
      c = 2 * math.asin( math.sqrt(a)) 
      r = 6371 # Radius of earth in kilometers.
      return c * r

def near(stop_row, lon, lat) :
    return haversine_distance(lon, lat, stop_row.stop_lat, stop_row.stop_lon)

In [73]:
import pandas
stops = pandas.read_csv('stops.txt', index_col = 0)

# Visualizing GeoSpacial Data

In this week's exercises we will explore a Python module called leaflet that allows us to visualize GeoSpacial data, i.e. data that has a longitude and latitude associated with it.

We will start by revisiting the static GTFS public transport data set that we explored in week 9 and will  extend it to fetching and visualizing real-time vehicle locations.

In [ ]:
# Let's start by using leaflet to create an interactive map
import ipyleaflet
map = ipyleaflet.Map()
map

# Zoom out until you can see some land and then navigate to Australia

In [ ]:
# Start as you did in week 9 by determining the exact longitute and latitude of the property where you live.
my_longitude = -35.351177
my_latitude = 149.233002

In [ ]:
# Now let's change the centre of the map, change the zoom level and change the map width and height
map = ipyleaflet.Map(center=(my_longitude, my_latitude), zoom=15)
map.layout.height="700px"
map.layout.width="1000px"
map

In [ ]:
# Add a marker to the map to show the location of your home

home = ipyleaflet.Marker(location=(my_longitude,my_latitude), draggable=False, icon=ipyleaflet.AwesomeIcon(name="home", marker_color='blue'), title="Home")
map.add_layer(home)

# Stops near me ...

The following is a repeat of what we did in the week 9 partical exercises

In [ ]:
def near(stop_row, lon, lat) :
    return haversine_distance(lon, lat, stop_row.stop_lat, stop_row.stop_lon)

stops['dist_from_home'] = stops.apply(near, lon=my_longitude, lat=my_latitude, axis=1)

nearby_stops = stops.sort_values('dist_from_home')
nearby_stops[:10]

# Visualize stops near me ...

In [ ]:
# We now want to visualize the 10 stops closest to our home.
# To do that, we will need to write a for loop that iterates through the rows in our Pandas dataframe:

for index,stop in nearby_stops[:10].iterrows() :
    print(index, stop.stop_name, stop.stop_lat, stop.stop_lon)

In [ ]:
# change the above loop so that it creates markers for each stop.
# change the marker location so that it is based on the longitude and latitude of each stop
# change the icon to show a "bus" rather than a home (https://fontawesome.com/v4/icons/)
# change the marker colour to green
# change the mouse over title to be the stop_id followed by the stop name

for index,stop in nearby_stops[:10].iterrows() :
    bus = ipyleaflet.Marker(location=(stop.stop_lat,stop.stop_lon), draggable=False, icon=ipyleaflet.AwesomeIcon(name="bus", marker_color='green'), title=stop.stop_name)
    map.add_layer(bus)

# Select a bus stop

In [ ]:
# When visualizing the stops, if you put your mouse over an icon it will show the title which includes the stop_id
# Select one of those stop_ids to explore further
our_stop_id = '262011'  # make sure it is expressed as a 'string' rather than as an integer (as some stop_ids are not numeric)

# Find buses departing from my stop soon ...

The following is a repeat of what we did in the week 9 partical exercises

In [ ]:
stop_times = pandas.read_csv('stop_times.txt', dtype={'stop_id':'str'})
services = pandas.read_csv('calendar.txt', index_col = 0, parse_dates=['start_date','end_date'])

In [ ]:
stop_times

In [ ]:
import pytz
timezone = pytz.timezone('Australia/Brisbane')
today = pandas.Timestamp.now(tz=timezone).tz_localize(None)

## Make sure your update the day in the following query to reflect the current day of the week ...

In [ ]:
todays_services = services[(services.thursday == 1) & (services.start_date <= today) & (today <= services.end_date)].index

In [ ]:
trips = pandas.read_csv('trips.txt', index_col = 2)

In [ ]:
todays_trips = trips[trips.service_id.isin(todays_services)].index

In [ ]:
time_now = today.strftime('%H:%M:%S')

arriving_soon = stop_times[(stop_times.stop_id==our_stop_id) & (stop_times.trip_id.isin(todays_trips)) & (time_now <= stop_times.arrival_time)  ]

In [ ]:
stops_with_trips = arriving_soon.join(trips, on='trip_id')

In [ ]:
routes = pandas.read_csv('routes.txt', index_col = 0)

In [ ]:
full = arriving_soon.join(trips, on='trip_id').join(routes, on='route_id')

In [ ]:
show = full[['trip_id','arrival_time', 'route_short_name', 'route_long_name', 'trip_headsign']]
show

## Select a trip

In [ ]:
# Select one of these trips to explore further ...
our_trip_id = '2061632'

In [ ]:
my_stops = stop_times[stop_times.trip_id == our_trip_id]
full_stop_data = my_stops.join(stops, on='stop_id')[['arrival_time', 'stop_name', 'stop_lat', 'stop_lon']]
full_stop_data

# Visualize the stops on the trip ...

In [ ]:
bus_markers = ipyleaflet.LayerGroup()
# We then add this layer group to the map
map.add_layer(bus_markers)

In [ ]:
# Add markers for each of these stops to your map.
# Use a different colour and icon from what you used previously for stops near you (https://fontawesome.com/v4/icons/)
# The mouse over title should be the arrival time followed  by the stop name
bus_markers.clear_layers()
for index,stop in full_stop_data.iterrows() :
    bus = ipyleaflet.Marker(location=(stop.stop_lat,stop.stop_lon), draggable=False, icon=ipyleaflet.AwesomeIcon(name="bus", marker_color='blue'), title=stop.stop_name)
    bus_markers.add_layer(bus)

In [71]:
map

Map(bottom=1269328.0, center=[-35.35118597263758, 149.23300266265872], controls=(ZoomControl(options=['positio…